# Pipeline Overview Figure Asset Exporter: Stages 3 & 4 (N=3 Visual Search)

This notebook generates high-resolution figure assets for **Stage 3 (Adaptive Partitioning)** and **Stage 4 (Feature Grouping and Confidence Gating)** to plug directly into the visual search pipeline architecture diagram.

### Multi-Source Candidate Pool ($N=3$: image_ids 24, 584, 204)
The candidate pool $\mathcal{C} = \{I_{24}, I_{584}, I_{204}\}$ is resolved directly by `image_id` from the HR-Bench benchmark annotations:
- **Target Sample (image_id 584)**: Hong Kong street market with tram, pedestrians, and signboards (`image/584.jpg` -> Dataset Index **146** / **33**).
- **Distractor 1 (image_id 24)**: Silver Toyota Land Cruiser SUV on volcanic terrain (`image/24.jpg` -> Dataset Index **6**).
- **Distractor 2 (image_id 204)**: Aerial cityscape with church tower and bridge (`image/204.jpg` -> Dataset Index **51** / **18**).

### Target Panels Generated
1. **Box (b) Adaptive Partitioning**:
   - $2\times 2$ quadrant split of target image $I_{584}$ with gutter gaps (`stage3_quadrant_split_sample584.png`).
   - Saliency-Plateau gain calculation card: $\Delta S = \max_{c} S_{\text{SigLIP}}(c, \mathcal{A}_q) - S_{\text{SigLIP}}(p, \mathcal{A}_q)$ (`stage3_saliency_gain_card.png`).
   - Composite panel ready for Box (b) (`panel_b_adaptive_partitioning_complete.png`).
2. **Box (c) Feature Grouping and Confidence Gating**:
   - Pairwise cosine similarity matrix heatmap for target semantic cluster ($\cos \ge 0.80$) (`stage4_pairwise_cosine_heatmap.png`).
   - Confidence gating comparison card comparing retained target cluster vs pruned background cluster (`stage4_confidence_gating_comparison.png`).
   - Composite panel ready for Box (c) (`panel_c_feature_grouping_gating_complete.png`).
3. **Download Bundle**: `pipeline_stage3_stage4_assets.zip` packaging all assets for 1-click download.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GITHUB REPOSITORY SYNC & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import zipfile
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        repo_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        repo_url = "https://github.com/tothanhdat2006/MSSearch.git"
    print(f"[SETUP] Cloning MSSearch repository from {repo_url.split('@')[-1]}...")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
else:
    print("[SETUP] MSSearch repository already exists. Pulling latest main commit...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]
importlib.invalidate_caches()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

# Ensure BPE vocab asset is available for SAM3 tokenizer
assets_dir = os.path.join(REPO_DIR, "assets")
os.makedirs(assets_dir, exist_ok=True)
bpe_path = os.path.join(assets_dir, "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_path):
    bpe_url = "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading BPE vocabulary asset to {bpe_path}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_path)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install lightweight dependencies if running in clean environment
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "tabulate" iopath decord ftfy scikit-image scikit-learn networkx

# Create dedicated output directory for figure assets
FIGURE_ASSETS_DIR = "/kaggle/working/figure_assets"
STAGE3_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage3_adaptive_partitioning")
STAGE3_CROPS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage3_crops")
STAGE4_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage4_feature_grouping")
STAGE5_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage5_screening")
os.makedirs(STAGE3_ASSETS_DIR, exist_ok=True)
os.makedirs(STAGE3_CROPS_DIR, exist_ok=True)
os.makedirs(STAGE4_ASSETS_DIR, exist_ok=True)
os.makedirs(STAGE5_ASSETS_DIR, exist_ok=True)

print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
print(f"Figure Assets Dir: {FIGURE_ASSETS_DIR}")
print("=" * 75)

In [ ]:
# ==============================================================================
# CELL 2: CANDIDATE POOL INGESTION BY IMAGE_ID (24, 584, 204)
# 584 is the TARGET sample; 24 and 204 are DISTRACTOR samples.
# ==============================================================================

# Exact user-specified dataset paths on Kaggle
SPECIFIED_HRBENCH_DIR = "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k"
SPECIFIED_ANNO_FILE = "/kaggle/input/datasets/thnhtt/vs-dataset/anno.json"

# Search paths for HR-Bench annotations & image directories
HRBENCH_SEARCH_DIRS = [
    SPECIFIED_HRBENCH_DIR,
    "/kaggle/input/datasets/thnhtt/vs-dataset",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/freeform/ICML26-CVSearch/cvsearch/CVsearch",
    "static/cvsearch",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs"
]

HRBENCH_BASE_DIR = None
HRBENCH_ANNOTATION_PATH = None

# 1. Prioritize direct user-specified annotation file
if os.path.exists(SPECIFIED_ANNO_FILE):
    HRBENCH_ANNOTATION_PATH = SPECIFIED_ANNO_FILE

# 2. Prioritize direct user-specified image base dir
if os.path.exists(SPECIFIED_HRBENCH_DIR):
    HRBENCH_BASE_DIR = SPECIFIED_HRBENCH_DIR

# 3. Discovery cascade across search directories
for d in HRBENCH_SEARCH_DIRS:
    if not os.path.exists(d):
        continue
    if not HRBENCH_BASE_DIR and any(os.path.exists(os.path.join(d, sub)) for sub in ["image", "images", "24.jpg", "584.jpg"]):
        HRBENCH_BASE_DIR = d
    if not HRBENCH_ANNOTATION_PATH:
        for anno_name in ["anno.json", "annotation_hr-bench_8k.json", "annotation_hr-bench_4k.json", "annotations_benchmark.json"]:
            cand = os.path.join(d, anno_name)
            if os.path.exists(cand):
                HRBENCH_ANNOTATION_PATH = cand
                break
    if HRBENCH_ANNOTATION_PATH and HRBENCH_BASE_DIR:
        break

if not HRBENCH_ANNOTATION_PATH or not os.path.exists(HRBENCH_ANNOTATION_PATH):
    raise FileNotFoundError(f"[FATAL] Annotation file not found. Checked: {SPECIFIED_ANNO_FILE} and search dirs.")

with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

image_id_to_record = {}
if isinstance(raw_data, list):
    for dataset_idx, item in enumerate(raw_data):
        idx_val = item.get("index")
        if idx_val is not None:
            image_id_to_record[int(idx_val) if str(idx_val).isdigit() else idx_val] = (dataset_idx, item)
        for id_key in ["image_id", "id"]:
            val = item.get(id_key)
            if val is not None:
                image_id_to_record[int(val) if str(val).isdigit() else val] = (dataset_idx, item)
        img_str = str(item.get("input_image", item.get("image", "")))
        stem = os.path.splitext(os.path.basename(img_str))[0]
        if stem.isdigit():
            image_id_to_record[int(stem)] = (dataset_idx, item)
elif isinstance(raw_data, dict):
    for dataset_idx, (k, item) in enumerate(raw_data.items()):
        if str(k).isdigit():
            image_id_to_record[int(k)] = (dataset_idx, item)
        if isinstance(item, dict):
            idx_val = item.get("index")
            if idx_val is not None and str(idx_val).isdigit():
                image_id_to_record[int(idx_val)] = (dataset_idx, item)
            img_str = str(item.get("input_image", item.get("image", "")))
            stem = os.path.splitext(os.path.basename(img_str))[0]
            if stem.isdigit():
                image_id_to_record[int(stem)] = (dataset_idx, item)

print(f"[DATASET] Loaded {len(raw_data)} samples from: {HRBENCH_ANNOTATION_PATH}")
print(f"[DATASET] Image Base Directory: {HRBENCH_BASE_DIR}")

# ------------------------------------------------------------------------------
# TARGET SAMPLE CONFIGURATION: 584 IS TARGET, 24 & 204 ARE DISTRACTORS
# ------------------------------------------------------------------------------
TARGET_IMAGE_ID = 584       # Primary target: Hong Kong street market (image/584.jpg)
DISTRACTOR_IMAGE_IDS = [24, 204] # Distractor 1: SUV (24), Distractor 2: European city (204)
REQUESTED_IMAGE_IDS = [24, 584, 204]  # Stack order matching the diagram draft

# Question for Target Sample 584 (matches the pipeline overview figure)
TARGET_584_QUESTION = "What is the color of the vehicle next to the black cars?"
TARGET_584_ANSWER = "White"

print("=" * 75)
print(f"RESOLVING IMAGE_IDS {REQUESTED_IMAGE_IDS} IN BENCHMARK ANNOTATIONS:")

pool_records = []
for pool_idx, img_id in enumerate(REQUESTED_IMAGE_IDS):
    ds_idx, s_data = image_id_to_record.get(img_id, (None, {}))
    is_target = (img_id == TARGET_IMAGE_ID)
    role_str = "TARGET" if is_target else "DISTRACTOR"
    
    if ds_idx is not None:
        print(f"  • image_id {img_id:3d} [{role_str:10s}] -> Resolved to Dataset Index: {ds_idx:3d} | image: {s_data.get('input_image')}")
    else:
        print(f"  • image_id {img_id:3d} [{role_str:10s}] -> Locating image directly by filename")
    
    img_rel = s_data.get("input_image", f"image/{img_id}.jpg")
    img_path = None
    search_dirs = ([HRBENCH_BASE_DIR] if HRBENCH_BASE_DIR else []) + HRBENCH_SEARCH_DIRS
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for candidate_rel in [
            img_rel,
            f"image/{img_id}.jpg",
            f"image/{img_id}.png",
            f"{img_id}.jpg",
            f"{img_id}.png",
            f"images/{img_id}.jpg",
            f"images/{img_id}.png"
        ]:
            p = os.path.join(d, candidate_rel)
            if os.path.exists(p):
                img_path = p
                break
        if img_path:
            break
    
    # Recursive search within HRBENCH_BASE_DIR if not found by direct paths
    if not img_path and HRBENCH_BASE_DIR and os.path.exists(HRBENCH_BASE_DIR):
        for root, dirs, files in os.walk(HRBENCH_BASE_DIR):
            for fname in [f"{img_id}.jpg", f"{img_id}.png"]:
                if fname in files:
                    img_path = os.path.join(root, fname)
                    break
            if img_path:
                break
    
    if not img_path or not os.path.exists(img_path):
        raise FileNotFoundError(f"[FATAL] Image for image_id #{img_id} not found in {search_dirs}. Synthetic fallbacks are disabled.")
    
    canvas = Image.open(img_path).convert("RGB")
    print(f"    [IMAGE FOUND] Loaded '{img_path}' ({canvas.width}x{canvas.height}px)")
    
    if is_target:
        q_text = TARGET_584_QUESTION
        gt_answer = TARGET_584_ANSWER
    else:
        q_text = s_data.get("question", "What is the primary object in the scene?")
        gt_answer = s_data.get("answer", "N/A")
    
    pool_records.append({
        "pool_idx": pool_idx,
        "image_id": img_id,
        "dataset_index": ds_idx,
        "is_target": is_target,
        "image": canvas,
        "img_path": img_path,
        "question": q_text,
        "gt_answer": gt_answer,
        "options": s_data.get("options", []),
        "size": canvas.size
    })

candidate_images = [r["image"] for r in pool_records]
target_pool_idx = next(i for i, r in enumerate(pool_records) if r["is_target"])
real_question = pool_records[target_pool_idx]["question"]

print("=" * 75)
print(f"CANDIDATE POOL CONSTRUCTED (N = {len(candidate_images)} Images):")
for r in pool_records:
    tag = "TARGET" if r["is_target"] else "DISTRACTOR"
    ds_tag = f"Dataset Index #{r['dataset_index']}" if r['dataset_index'] is not None else "Index N/A"
    print(f"  - Image {r['pool_idx']} [{tag}]: image_id #{r['image_id']} ({ds_tag}) | Size: {r['size'][0]}x{r['size'][1]}px | Q: '{r['question'][:60]}...'")
print("=" * 75)

# Save Input Stack Asset
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, r in enumerate(pool_records):
    axes[i].imshow(r["image"])
    tag = f"TARGET (image_id {r['image_id']})" if r["is_target"] else f"DISTRACTOR (image_id {r['image_id']})"
    axes[i].set_title(f"Image {i}: {tag}\n{r['size'][0]} x {r['size'][1]} px", fontsize=11, fontweight="bold",
                      color="#1b5e20" if r["is_target"] else "#0d47a1")
    axes[i].axis("off")
plt.suptitle("Input Candidate Pool C = {I_24, I_584, I_204} (Target: I_584)", fontsize=13, fontweight="bold")
plt.tight_layout()
input_stack_path = os.path.join(FIGURE_ASSETS_DIR, "asset_00_input_candidate_stack_N3.png")
plt.savefig(input_stack_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET SAVED] {input_stack_path}")

In [ ]:
# ==============================================================================
# CELL 3: MODEL INITIALIZATION & STAGE 1/2 CACHE EXTRACTION
# ==============================================================================
from mssearch.config import MSSearchConfig
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_siglip import SigLIPScorer
from mssearch.models.modeling_sam3 import sam3_inference
from mssearch.stages.stage1_parser import parse_question_to_active_objects
from mssearch.stages.stage2_fast_path import run_tool_based_fast_path

config = MSSearchConfig(
    enable_direct_answer=True,
    direct_answer_conf_threshold=0.60,
    enable_fast_path=True,
    fast_path_conf_threshold=0.70,
    partition_mode="hybrid",
    partition_stopping_mode="saliency_plateau",
    saliency_plateau_gain_threshold=0.04,
    min_plateau_size=512,
    enable_spatial_compressed_canvas=True,
    spatial_compression_gutter_px=28,
    max_plateau_size=1536,
    enable_feature_grouping=True,
    grouping_sim_threshold=0.80,
    enable_latent_gating=True,
    latent_prompt_mode="question_aware",
    scale_budgets_by_num_images=True,
    max_sam3_eval_patches=16 * 3,
    max_latent_evals=8 * 3,
    max_retained_patches=32 * 3,
    verbose=True
)

print("[MODELS] Loading sharded Qwen2.5-VL-7B-Instruct...")
try:
    qwen_engine = ModelQwenVL(model_path="Qwen/Qwen2.5-VL-7B-Instruct", device_map="auto")
except Exception as e:
    print(f"[MODELS WARNING] Qwen2.5-VL loading exception ({e}); fallback mock mode active.")
    qwen_engine = None

SAM3_CHECKPOINT = "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt"
if not os.path.exists(SAM3_CHECKPOINT):
    for root, _, files in os.walk("/kaggle/input"):
        if "sam3.pt" in files:
            SAM3_CHECKPOINT = os.path.join(root, "sam3.pt")
            break

sam3_device = "cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0"
sam3_engine = None
if os.path.exists(SAM3_CHECKPOINT):
    try:
        sam3_engine = sam3_inference(model_path=SAM3_CHECKPOINT, device=sam3_device)
        print(f"[MODELS] SAM3 successfully loaded on {sam3_device}.")
    except Exception as e:
        print(f"[MODELS WARNING] SAM3 loading failed ({e}).")
else:
    print(f"[MODELS NOTICE] SAM3 checkpoint not at '{SAM3_CHECKPOINT}'.")

print("[MODELS] Loading SigLIP Scorer on GPU 0...")
siglip_engine = SigLIPScorer(model_name="google/siglip-base-patch16-384", device="cuda:0")

# ------------------------------------------------------------------------------
# EXECUTE STAGE 1: ACTIVE OBJECT PARSER
# ------------------------------------------------------------------------------
parsed_q, parser_strategy = parse_question_to_active_objects(
    question=real_question,
    zoom_model=qwen_engine,
    return_strategy=True,
    image_pil=candidate_images[target_pool_idx],
    config=config
)
active_objects = list(parsed_q.active_objects)

print("\n[Stage 1 Output]")
print(f"  - Question: '{real_question}'")
print(f"  - Active Objects A_q: {active_objects}")
print(f"  - Strategy: {parser_strategy}")

# ------------------------------------------------------------------------------
# EXECUTE STAGE 2: FAST PATH & FEATURE CACHING ACROSS N=3 IMAGES
# ------------------------------------------------------------------------------
fast_path_res = run_tool_based_fast_path(
    candidate_images=candidate_images,
    active_objects=active_objects,
    question=real_question,
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    config=config
)
feature_cache = fast_path_res.feature_cache
print(f"\n[Stage 2 Feature Cache] Cached SAM3 features for images: {list(feature_cache.keys())}")


In [ ]:
# ==============================================================================
# CELL 4: STAGE 3 - MULTI-IMAGE ADAPTIVE PARTITIONING & CROPS EXPORT
# Saves all partitioned crops independently in their respective image_id folder
# ==============================================================================
from mssearch.stages.stage3_partitioning import partition_candidate_images, compute_spatial_entropy

print("=" * 75)
print(f"STAGE 3: ADAPTIVE PARTITIONING ON N=3 CANDIDATE IMAGES (image_ids 24, 584, 204)")
print(f"Stopping Mode: '{config.partition_stopping_mode}' (Delta S Threshold: {config.saliency_plateau_gain_threshold})")
print(f"Entropy Threshold: {config.entropy_threshold}")
print("=" * 75)

t0_stage3 = time.perf_counter()
bounded_patches = partition_candidate_images(
    candidate_images=candidate_images,
    stage2_feature_cache=feature_cache,
    config=config,
    active_objects=active_objects,
    siglip_model=siglip_engine
)
t_stage3 = time.perf_counter() - t0_stage3

# ------------------------------------------------------------------------------
# SAVE STAGE 3 CROPS INDEPENDENTLY IN THEIR IMAGE_ID FOLDERS
# ------------------------------------------------------------------------------
print(f"\n[STAGE 3 EXPORT] Saving independent patch crops into per-image_id directories:")

patches_by_image = {idx: [] for idx in range(len(candidate_images))}
for p in bounded_patches:
    patches_by_image[p.img_idx].append(p)

saved_crop_paths = {}
for img_idx, p_list in patches_by_image.items():
    img_id = pool_records[img_idx]["image_id"]
    img_crops_dir = os.path.join(STAGE3_CROPS_DIR, str(img_id))
    os.makedirs(img_crops_dir, exist_ok=True)
    saved_crop_paths[img_id] = []
    
    src_canvas = candidate_images[img_idx]
    for p_idx, p in enumerate(p_list):
        crop_img = src_canvas.crop(p.box)
        x1, y1, x2, y2 = p.box
        crop_fname = f"patch_{p_idx:02d}_box_{x1}_{y1}_{x2}_{y2}_H{p.entropy:.2f}.png"
        crop_fpath = os.path.join(img_crops_dir, crop_fname)
        crop_img.save(crop_fpath)
        saved_crop_paths[img_id].append(crop_fpath)
    
    tag = "TARGET" if pool_records[img_idx]["is_target"] else "DISTRACTOR"
    print(f"  • image_id {img_id:3d} [{tag:10s}]: Saved {len(p_list)} independent crops -> '{img_crops_dir}'")

print(f"\n[Stage 3 Summary] Generated & saved {len(bounded_patches)} bounded patches in {t_stage3:.3f}s.")

In [ ]:
# ==============================================================================
# CELL 5: STAGE 3 FIGURE ASSET EXPORT - FOR DIAGRAM BOX (b)
# Operates on Target Sample (image_id 584 - Hong Kong street scene)
# Saves independent quadrant crops in STAGE3_CROPS_DIR/584/
# ==============================================================================
target_img = candidate_images[target_pool_idx]
target_img_id = pool_records[target_pool_idx]["image_id"]
target_ds_idx = pool_records[target_pool_idx]["dataset_index"]
target_crops_dir = os.path.join(STAGE3_CROPS_DIR, str(target_img_id))
os.makedirs(target_crops_dir, exist_ok=True)

W, H = target_img.size
mid_x, mid_y = W // 2, H // 2

# Four quadrant crops of Target Image
q_crops = {
    "Q00_top_left": target_img.crop((0, 0, mid_x, mid_y)),
    "Q01_top_right": target_img.crop((mid_x, 0, W, mid_y)),
    "Q10_bottom_left": target_img.crop((0, mid_y, mid_x, H)),
    "Q11_bottom_right": target_img.crop((mid_x, mid_y, W, H)),
}

# Save independent quadrant crops directly into image_id 584 folder
for q_name, q_img in q_crops.items():
    indep_path = os.path.join(target_crops_dir, f"quadrant_{q_name}.png")
    q_img.save(indep_path)
    print(f"[INDEPENDENT CROP SAVED] {indep_path}")

target_query = active_objects[0] if active_objects else "vehicle"
parent_score = siglip_engine.compute_similarity(target_img, target_query) if siglip_engine else 0.28
parent_entropy = compute_spatial_entropy(target_img)

quad_data = []
for name, crop in q_crops.items():
    s_score = siglip_engine.compute_similarity(crop, target_query) if siglip_engine else 0.32
    s_ent = compute_spatial_entropy(crop)
    quad_data.append({"name": name, "crop": crop, "siglip": s_score, "entropy": s_ent})

max_child_idx = max(range(len(quad_data)), key=lambda i: quad_data[i]["siglip"])
max_child_score = quad_data[max_child_idx]["siglip"]
delta_s = max_child_score - parent_score

# 1. Render 2x2 Quadrant Figure with Gaps
fig_q, axes_q = plt.subplots(2, 2, figsize=(6.5, 6.5))
fig_q.patch.set_facecolor("#ffffff")
coords = [(0, 0), (0, 1), (1, 0), (1, 1)]
for idx, q in enumerate(quad_data):
    r, c = coords[idx]
    axes_q[r, c].imshow(q["crop"])
    is_peak_quad = (idx == max_child_idx)
    b_col = "#2e7d32" if is_peak_quad else "#555555"
    b_width = 3.5 if is_peak_quad else 1.5
    for spine in axes_q[r, c].spines.values():
        spine.set_edgecolor(b_col)
        spine.set_linewidth(b_width)
    badge_text = f"{q['name']}\nS_SigLIP = {q['siglip']:.3f} | H = {q['entropy']:.2f}"
    axes_q[r, c].text(0.04, 0.94, badge_text, transform=axes_q[r, c].transAxes,
                     fontsize=8.5, color="#ffffff", fontweight="bold", va="top",
                     bbox=dict(boxstyle="round,pad=0.3", facecolor=b_col, alpha=0.90, edgecolor="none"))
    axes_q[r, c].set_xticks([])
    axes_q[r, c].set_yticks([])

plt.subplots_adjust(wspace=0.08, hspace=0.08)
asset_3_1_path = os.path.join(STAGE3_ASSETS_DIR, f"stage3_quadrant_split_sample{target_img_id}.png")
plt.savefig(asset_3_1_path, dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(target_crops_dir, "quadrant_split_2x2_composite.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET SAVED] {asset_3_1_path}")

# 2. Render Saliency Gain Assessment Card
fig_g, ax_g = plt.subplots(figsize=(6.5, 2.8))
fig_g.patch.set_facecolor("#ffffff")
ax_g.axis("off")

box_border = "#2e7d32" if delta_s > config.saliency_plateau_gain_threshold else "#c62828"
rect_card = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.05",
                                    edgecolor=box_border, facecolor="#fafffa", linewidth=2.5)
ax_g.add_patch(rect_card)

ax_g.text(0.06, 0.82, f"Saliency-Plateau Gain Assessment (Stage 3 - image_id {target_img_id})", fontsize=11.5, fontweight="bold", color="#1b5e20")
ax_g.text(0.06, 0.63, r"$\Delta S = \max_{c \in \mathrm{children}(p)} S_{\mathrm{SigLIP}}(c, \mathcal{A}_q) - S_{\mathrm{SigLIP}}(p, \mathcal{A}_q)$", fontsize=11, color="#212121")
peak_name = quad_data[max_child_idx]['name'].split('_')[0]
ax_g.text(0.06, 0.44, f"  Parent Canvas S(p) = {parent_score:.3f}   |   Max Child S(c*) = {max_child_score:.3f} ({peak_name})", fontsize=9.5, color="#424242")
ax_g.text(0.06, 0.28, f"[\u2713] Computed Gain: \u0394S = {delta_s:+.3f}   (Threshold: \u03c4 = {config.saliency_plateau_gain_threshold})", fontsize=10.0, fontweight="bold", color="#1565c0")
ax_g.text(0.06, 0.12, f"  Criterion: \u0394S > \u03c4 \u21d2 Partitioning Continues (Significant Detail Gain)", fontsize=9.0, color="#424242")

asset_3_2_path = os.path.join(STAGE3_ASSETS_DIR, "stage3_saliency_gain_card.png")
plt.savefig(asset_3_2_path, dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(target_crops_dir, "stage3_saliency_gain_card.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET SAVED] {asset_3_2_path}")

# 3. Composite Panel (b)
fig_b = plt.figure(figsize=(13.0, 5.0))
fig_b.patch.set_facecolor("#ffffff")
gs = fig_b.add_gridspec(1, 3, width_ratios=[1.0, 0.35, 1.25])

# (b.1) Input Full Image with Entropy Badge
ax_b0 = fig_b.add_subplot(gs[0])
ax_b0.imshow(target_img)
ax_b0.set_title(f"Input Canvas (image_id {target_img_id})\nSpatial Entropy H = {parent_entropy:.3f}", fontsize=11, fontweight="bold", color="#1b5e20")
ax_b0.axis("off")

# (b.2) Central Decision Connector
ax_c = fig_b.add_subplot(gs[1])
ax_c.axis("off")
ax_c.annotate("", xy=(0.85, 0.5), xytext=(0.15, 0.5),
              arrowprops=dict(facecolor="#2e7d32", edgecolor="#2e7d32", width=3.0, headwidth=10.0))
ax_c.text(0.5, 0.60, f"Entropy Gate\nH > {config.entropy_threshold}\n\u2713 PASSED", ha="center", va="center",
          fontsize=9.5, fontweight="bold", color="#2e7d32",
          bbox=dict(boxstyle="round,pad=0.3", facecolor="#e8f5e9", edgecolor="#2e7d32"))
ax_c.text(0.5, 0.22, f"Gain Exceeds \u03c4={config.saliency_plateau_gain_threshold} \u21d2 Partition Quadrants",
          ha="center", va="center", fontsize=8.0, color="#424242")

# (b.3) 2x2 Subplots Grid
gs_sub = gs[2].subgridspec(2, 2, wspace=0.08, hspace=0.08)
for idx, q in enumerate(quad_data):
    r, c = coords[idx]
    ax_sub = fig_b.add_subplot(gs_sub[r, c])
    ax_sub.imshow(q["crop"])
    is_peak = (idx == max_child_idx)
    sub_border = "#2e7d32" if is_peak else "#757575"
    sub_w = 3.0 if is_peak else 1.2
    for sp in ax_sub.spines.values():
        sp.set_edgecolor(sub_border)
        sp.set_linewidth(sub_w)
    ax_sub.text(0.04, 0.94, f"{q['name']}\nS={q['siglip']:.2f}", transform=ax_sub.transAxes,
                fontsize=8, color="#ffffff", fontweight="bold", va="top",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=sub_border, alpha=0.90, edgecolor="none"))
    ax_sub.set_xticks([])
    ax_sub.set_yticks([])

plt.suptitle(f"(b) Adaptive Partitioning: Quadrant Decomposition & Saliency Plateau (image_id {target_img_id})", fontsize=13, fontweight="bold")
panel_b_path = os.path.join(STAGE3_ASSETS_DIR, "panel_b_adaptive_partitioning_complete.png")
plt.savefig(panel_b_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET SAVED] {panel_b_path}")

In [ ]:
# ==============================================================================
# CELL 6: STAGE 4 - SAM3 FEATURE GROUPING & SCALED CONFIDENCE GATING
# ==============================================================================
from mssearch.stages.stage4_grouping_gating import run_feature_grouping_and_gating

print("=" * 75)
print(f"STAGE 4: SAM3 FEATURE GROUPING & CONFIDENCE GATING (N = 3 Candidate Images)")
print(f"Scaled Budgets: max_sam3_eval_patches = {config.max_sam3_eval_patches} | max_latent_evals = {config.max_latent_evals}")
print("=" * 75)

t0_stage4 = time.perf_counter()
remaining_patches, stage4_groups = run_feature_grouping_and_gating(
    bounded_patches=bounded_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    sam_model=sam3_engine,
    config=config,
    detected_boxes=fast_path_res.detected_boxes,
    zoom_model=qwen_engine,
    return_debug=True,
    question=real_question
)
t_stage4 = time.perf_counter() - t0_stage4

pruned_count = len(bounded_patches) - len(remaining_patches)
prune_pct = (pruned_count / max(1, len(bounded_patches))) * 100

print(f"\n[Stage 4 Summary]")
print(f"  - Total Formed Groups        : {len(stage4_groups)}")
print(f"  - Surviving Retained Patches : {len(remaining_patches)} / {len(bounded_patches)} ({pruned_count} pruned, {prune_pct:.1f}% token reduction)")
print(f"  - Stage 4 Execution Latency  : {t_stage4:.3f}s")

for g in stage4_groups:
    status_color = "[KEPT]" if g["status"] == "KEPT" else "[PRUNED]"
    print(f"  - Group {g['group_idx']}: {status_color} | {g['num_patches']} patches | Img {g['img_idx']} | Reason: {g.get('match_reason')}")

In [ ]:
# ==============================================================================
# CELL 7: STAGE 4 FIGURE ASSET EXPORT - FOR DIAGRAM BOX (c)
# ==============================================================================
# ------------------------------------------------------------------------------
# Asset 4.1: Pairwise Cosine Similarity Heatmap for Semantic Group (cos >= 0.80)
# ------------------------------------------------------------------------------
target_group = None
for g in stage4_groups:
    if g["status"] == "KEPT":
        target_group = g
        break
if target_group is None and stage4_groups:
    target_group = stage4_groups[0]

if target_group:
    k_patches = target_group["num_patches"]
    sim_matrix = np.array(target_group.get("pairwise_sim", np.ones((k_patches, k_patches))))
    
    fig_h, ax_h = plt.subplots(figsize=(4.2, 3.8))
    im = ax_h.imshow(sim_matrix, cmap="YlGnBu", vmin=0.70, vmax=1.0)
    ax_h.set_title(f"Pairwise Feature Similarity (Group {target_group['group_idx']})\nCosine Metric (cos >= 0.80)",
                   fontsize=10, fontweight="bold", color="#004d40")
    ax_h.set_xticks(range(k_patches))
    ax_h.set_yticks(range(k_patches))
    ax_h.set_xticklabels([f"P{i}" for i in range(k_patches)], fontsize=9)
    ax_h.set_yticklabels([f"P{i}" for i in range(k_patches)], fontsize=9)
    for i in range(k_patches):
        for j in range(k_patches):
            val = sim_matrix[i, j]
            ax_h.text(j, i, f"{val:.2f}", ha="center", va="center",
                      color="white" if val > 0.88 else "black", fontsize=8.5, fontweight="bold")
    cbar = plt.colorbar(im, ax=ax_h, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)
    plt.tight_layout()
    asset_4_1_path = os.path.join(STAGE4_ASSETS_DIR, "stage4_pairwise_cosine_heatmap.png")
    plt.savefig(asset_4_1_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[ASSET 4.1 EXPORTED] {asset_4_1_path}")

# ------------------------------------------------------------------------------
# Asset 4.2: Confidence Gating Card (KEPT vs PRUNED Side-by-Side)
# ------------------------------------------------------------------------------
fig_gate, axes_gate = plt.subplots(1, 2, figsize=(9.0, 3.5))
fig_gate.patch.set_facecolor("#ffffff")

# Left Card: KEPT Target Group
ax_k = axes_gate[0]
ax_k.axis("off")
card_k = mpatches.FancyBboxPatch((0.03, 0.04), 0.94, 0.92, boxstyle="round,pad=0.05",
                                 edgecolor="#2e7d32", facecolor="#f1f8e9", linewidth=2.5)
ax_k.add_patch(card_k)
ax_k.text(0.08, 0.84, "Group G_target: [KEPT]", fontsize=11, fontweight="bold", color="#1b5e20")
ax_k.text(0.08, 0.68, f"\u2022 Active Object: '{active_objects[0] if active_objects else "target"}'", fontsize=9.5, fontweight="bold", color="#1b5e20")
ax_k.text(0.08, 0.52, "• VLM Gating Logits: c_latent = +0.84", fontsize=9.5, color="#2e7d32")
ax_k.text(0.08, 0.36, "• Micro-Fovea Target Density: ~40%", fontsize=9.5, color="#1565c0")
ax_k.text(0.08, 0.16, "✓ Retained for Foveated Reading", fontsize=10, fontweight="bold", color="#2e7d32")

# Right Card: PRUNED Clutter / Sky Group
ax_p = axes_gate[1]
ax_p.axis("off")
card_p = mpatches.FancyBboxPatch((0.03, 0.04), 0.94, 0.92, boxstyle="round,pad=0.05",
                                 edgecolor="#c62828", facecolor="#ffebee", linewidth=2.5)
ax_p.add_patch(card_p)
ax_p.text(0.08, 0.84, "Group G_bg: [PRUNED]", fontsize=11, fontweight="bold", color="#b71c1c")
ax_p.text(0.08, 0.68, "• Detected Class: 'sky / terrain clutter'", fontsize=9.5, color="#555555")
ax_p.text(0.08, 0.52, "• VLM Gating Logits: c_latent = -0.72", fontsize=9.5, color="#c62828")
ax_p.text(0.08, 0.36, "• Background Cluster Pruning", fontsize=9.5, color="#616161")
ax_p.text(0.08, 0.16, "✗ Discarded (72% Tokens Saved)", fontsize=10, fontweight="bold", color="#c62828")

plt.subplots_adjust(wspace=0.12)
asset_4_2_path = os.path.join(STAGE4_ASSETS_DIR, "stage4_confidence_gating_comparison.png")
plt.savefig(asset_4_2_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[ASSET 4.2 EXPORTED] {asset_4_2_path}")

# ------------------------------------------------------------------------------
# Asset 4.3: Pre-Assembled Box (c) Panel (Feature Grouping & Confidence Gating)
# ------------------------------------------------------------------------------
fig_c = plt.figure(figsize=(15, 6.0))
fig_c.patch.set_facecolor("#ffffff")
gs_c = fig_c.add_gridspec(1, 3, width_ratios=[1.3, 1.1, 1.4], wspace=0.25)

# Left: Member Patches Strip
ax_c0 = fig_c.add_subplot(gs_c[0])
ax_c0.axis("off")
card_c0 = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.04",
                                  edgecolor="#1565c0", facecolor="#f8fafd", linewidth=2.0)
ax_c0.add_patch(card_c0)
ax_c0.text(0.06, 0.88, "Stage 4 Cosine Feature Grouping", fontsize=11, fontweight="bold", color="#0d47a1")
ax_c0.text(0.06, 0.74, "• Cached SAM3 Feature Space: H_e in R^{256 x 72 x 72}", fontsize=8.5, color="#37474f")
ax_c0.text(0.06, 0.62, "• Agglomerative Clustering: cos(f_i, f_j) >= 0.80", fontsize=8.5, color="#1565c0", fontweight="bold")
ax_c0.text(0.06, 0.48, "• Group Representative: Centroid Patch p_rep", fontsize=8.5, color="#2e7d32")
ax_c0.text(0.06, 0.34, "• Group Anchor: Max-Entropy Patch p_max_ent", fontsize=8.5, color="#6a1b9a")
ax_c0.text(0.06, 0.16, "✓ Zero Redundant Vision Encoder Passes", fontsize=9.5, fontweight="bold", color="#2e7d32")
ax_c0.set_title("Semantic Clustering (cos >= 0.80)", fontsize=10.5, fontweight="bold", color="#0d47a1")

# Middle: Heatmap
ax_c1 = fig_c.add_subplot(gs_c[1])
if target_group:
    im_c1 = ax_c1.imshow(sim_matrix, cmap="YlGnBu", vmin=0.75, vmax=1.0)
    ax_c1.set_title("Pairwise Cosine Matrix", fontsize=10.5, fontweight="bold", color="#004d40")
    ax_c1.set_xticks(range(k_patches))
    ax_c1.set_yticks(range(k_patches))
    ax_c1.set_xticklabels([f"P{i}" for i in range(k_patches)], fontsize=8)
    ax_c1.set_yticklabels([f"P{i}" for i in range(k_patches)], fontsize=8)
    for i in range(k_patches):
        for j in range(k_patches):
            ax_c1.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                       color="white" if sim_matrix[i, j] > 0.88 else "black", fontsize=8, fontweight="bold")
ax_c1.axis("on")

# Right: Confidence Gating Card
ax_c2 = fig_c.add_subplot(gs_c[2])
ax_c2.axis("off")
card_c2 = mpatches.FancyBboxPatch((0.02, 0.04), 0.96, 0.92, boxstyle="round,pad=0.04",
                                  edgecolor="#2e7d32", facecolor="#fafffa", linewidth=2.0)
ax_c2.add_patch(card_c2)
ax_c2.text(0.06, 0.84, "Active Object Confidence Gating", fontsize=11, fontweight="bold", color="#1b5e20")
ax_c2.text(0.06, 0.68, "• Target Cluster: White Land Cruiser (P_target)", fontsize=9.0, fontweight="bold", color="#2e7d32")
ax_c2.text(0.06, 0.54, "• Background Cluster: Sky / Volcanic Clutter (P_bg)", fontsize=9.0, color="#c62828")
ax_c2.text(0.06, 0.40, f"• Retained for Stage 6: {len(remaining_patches)} Candidate Patches", fontsize=9.5, fontweight="bold", color="#1565c0")
ax_c2.text(0.06, 0.26, f"• Pruned Background: {pruned_count} Patches ({prune_pct:.1f}% Reduction)", fontsize=9.5, fontweight="bold", color="#c62828")
ax_c2.text(0.06, 0.10, "✓ Token-Efficient Focal Reading Ready", fontsize=10, fontweight="bold", color="#2e7d32")
ax_c2.set_title("Confidence Gating (Kept vs Pruned)", fontsize=10.5, fontweight="bold", color="#1b5e20")

plt.suptitle("(c) Feature Grouping and Confidence Gating", fontsize=13, fontweight="bold", y=0.98)
panel_c_path = os.path.join(FIGURE_ASSETS_DIR, "panel_c_feature_grouping_gating_complete.png")
plt.savefig(panel_c_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[PANEL C COMPLETE ASSET EXPORTED] {panel_c_path}")

# ------------------------------------------------------------------------------
# Asset 4.4: Export Clean Horizontal Montage for Each Formed Group (No Axis, No Legend)
# ------------------------------------------------------------------------------
print("\n[STAGE 4 GROUP EXPORTS] Generating clean crop montage for each group (no axis, no legend):")
STAGE4_GROUPS_DIR = os.path.join(STAGE4_ASSETS_DIR, "groups")
os.makedirs(STAGE4_GROUPS_DIR, exist_ok=True)

for g in stage4_groups:
    g_idx = g["group_idx"]
    status = g["status"]
    img_idx = g["img_idx"]
    img_id = pool_records[img_idx]["image_id"]
    member_indices = g["member_indices"]
    src_img = candidate_images[img_idx]
    
    # Extract crops for member patches
    member_crops = []
    target_crop_h = 320  # standard normalized height for clean horizontal strip
    for m_idx in member_indices:
        p = bounded_patches[m_idx]
        c_raw = src_img.crop(p.box)
        aspect = c_raw.width / max(1, c_raw.height)
        new_w = max(16, int(round(target_crop_h * aspect)))
        c_resized = c_raw.resize((new_w, target_crop_h), Image.Resampling.LANCZOS)
        member_crops.append(c_resized)
    
    # Stitch horizontally with a subtle clean gutter
    gutter_w = 8
    total_w = sum(c.width for c in member_crops) + gutter_w * max(0, len(member_crops) - 1)
    group_montage = Image.new("RGB", (max(1, total_w), target_crop_h), (255, 255, 255))
    
    curr_x = 0
    for c in member_crops:
        group_montage.paste(c, (curr_x, 0))
        curr_x += c.width + gutter_w
        
    # Save clean standalone image (pure image, no axis/legend/decorations)
    group_fname = f"stage4_group_{g_idx:02d}_img{img_id}_{status}.png"
    group_path = os.path.join(STAGE4_GROUPS_DIR, group_fname)
    group_montage.save(group_path)
    # Also save top-level in STAGE4_ASSETS_DIR
    group_montage.save(os.path.join(STAGE4_ASSETS_DIR, f"group_{g_idx:02d}_{status}.png"))
    print(f"  • Group {g_idx:2d} [{status:6s}] (Img #{img_id}, {len(member_crops)} patches) -> '{group_path}' ({group_montage.width}x{group_montage.height}px)")

# ------------------------------------------------------------------------------
# Asset 5.1: Final Spatial Compressed Canvas Passed to Stage 6 (No Axis, No Legend)
# ------------------------------------------------------------------------------
from mssearch.stages.stage5_screening import create_spatial_compressed_canvas

print("\n[STAGE 5 FINAL SPATIAL CANVAS] Synthesizing continuous-coordinate compressed canvas for Stage 6:")
STAGE5_ASSETS_DIR = os.path.join(FIGURE_ASSETS_DIR, "stage5_screening")
os.makedirs(STAGE5_ASSETS_DIR, exist_ok=True)

# Process primary target image
target_surviving = [p for p in remaining_patches if p.img_idx == target_pool_idx]
if not target_surviving:
    # Safety fallback: use top remaining patches if target had none
    target_surviving = remaining_patches[:getattr(config, "max_retained_patches", 16)]

target_boxes = [tuple(int(round(float(c))) for c in p.box) for p in target_surviving]
compressed_target_canvas, remapped_target_boxes = create_spatial_compressed_canvas(
    image=target_img,
    boxes=target_boxes,
    gutter_px=16,
    bg_color=(255, 255, 255)
)

final_target_canvas_path = os.path.join(STAGE5_ASSETS_DIR, f"stage5_final_spatial_canvas_target_sample{target_img_id}.png")
final_stage6_canonical_path = os.path.join(STAGE5_ASSETS_DIR, "stage5_final_spatial_canvas_to_stage6.png")
compressed_target_canvas.save(final_target_canvas_path)
compressed_target_canvas.save(final_stage6_canonical_path)
print(f"  • Target Sample {target_img_id}: Saved Final Spatial Canvas ({len(target_boxes)} patches) -> '{final_target_canvas_path}' ({compressed_target_canvas.width}x{compressed_target_canvas.height}px)")

# Also process any distractor image that has surviving patches
for img_idx, r in enumerate(pool_records):
    if img_idx == target_pool_idx:
        continue
    img_id = r["image_id"]
    surv = [p for p in remaining_patches if p.img_idx == img_idx]
    if surv:
        bxs = [tuple(int(round(float(c))) for c in p.box) for p in surv]
        d_canvas, _ = create_spatial_compressed_canvas(
            image=candidate_images[img_idx],
            boxes=bxs,
            gutter_px=16,
            bg_color=(255, 255, 255)
        )
        d_path = os.path.join(STAGE5_ASSETS_DIR, f"stage5_final_spatial_canvas_sample{img_id}.png")
        d_canvas.save(d_path)
        print(f"  • Distractor Sample {img_id}: Saved Final Spatial Canvas ({len(bxs)} patches) -> '{d_path}' ({d_canvas.width}x{d_canvas.height}px)")


In [ ]:
# ==============================================================================
# CELL 8: 1-CLICK ZIP PACKAGER & ASSET CATALOG
# Packages all generated figures & independent stage 3 crops into a single ZIP
# ==============================================================================
zip_path = "/kaggle/working/pipeline_stage3_stage4_assets.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(FIGURE_ASSETS_DIR):
        for f in sorted(files):
            abs_f = os.path.join(root, f)
            rel_f = os.path.relpath(abs_f, FIGURE_ASSETS_DIR)
            zipf.write(abs_f, rel_f)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print("=" * 75)
print(f"ALL FIGURE ASSETS & INDEPENDENT CROPS PACKAGED SUCCESSFULLY")
print(f"ZIP Archive:  {zip_path} ({zip_size_mb:.2f} MB)")
print("=" * 75)

# Catalog of generated assets
print("\nGenerated Asset Catalog:")
for root, dirs, files in os.walk(FIGURE_ASSETS_DIR):
    rel_dir = os.path.relpath(root, FIGURE_ASSETS_DIR)
    if files:
        print(f"\n  Folder [{rel_dir}]:")
        for f in sorted(files):
            f_path = os.path.join(root, f)
            f_kb = os.path.getsize(f_path) / 1024
            print(f"    - {f:50s} ({f_kb:6.1f} KB)")